In [ ]:
# check cuda
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using GPU.")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")


In [ ]:
# Check current Python environment and system info
import sys
import os
import socket

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"Hostname: {socket.gethostname()}")
print(f"Current working directory: {os.getcwd()}")
print("\nPython path:")
for path in sys.path[:5]:
    print(f"  {path}")

# DPO DATA

In [ ]:
from datasets import load_dataset

In [ ]:
DATASET_ID = "HuggingFaceH4/ultrafeedback_binarized"
dataset = load_dataset(path=DATASET_ID, split="train_sft")

In [ ]:
def preprocess_function(examples):
    # examples['prompt'] is a list of lists of messages (dicts with 'role' and 'content')
    processed = {
        "prompt_input_ids": [],
        "chosen_input_ids": [],
        "rejected_input_ids": [],
        "prompt_attention_mask": [],
        "chosen_attention_mask": [],
        "rejected_attention_mask": [],
        "prompt_len": []
    }

    for i in range(len(examples['prompt'])):
        current_prompt_messages = examples['prompt'][i]
        current_chosen_messages = examples['chosen'][i]
        current_rejected_messages = examples['rejected'][i]

        # Helper to validate and potentially fix message lists
        def ensure_message_list(messages, is_prompt=False, idx=i):
            if isinstance(messages, list) and all(isinstance(m, dict) and 'role' in m and 'content' in m for m in messages):
                return messages
            elif isinstance(messages, str):
                # If it's a string, try to wrap it as a simple user message.
                # This is a heuristic for malformed data; assumes the string is the user's input.
                if is_prompt:
                    # print(f"Warning: Prompt entry {idx} is a string. Wrapping as 'user' message.")
                    return [{"role": "user", "content": messages}]
                else: # Chosen/rejected responses should not be simple strings
                    # print(f"Warning: Chosen/Rejected entry {idx} is a string, which is unexpected. Skipping example.")
                    return None
            else:
                # If it's not a list or a string, it's an unrecognized format.
                print(f"Warning: Malformed entry {idx} (type: {type(messages)}). Skipping example.")
                return None

        current_prompt_messages = ensure_message_list(current_prompt_messages, is_prompt=True)
        current_chosen_messages = ensure_message_list(current_chosen_messages)
        current_rejected_messages = ensure_message_list(current_rejected_messages)

        if current_prompt_messages is None or current_chosen_messages is None or current_rejected_messages is None:
            continue # Skip this example if any part is malformed


        # Format prompt for DPO: add empty assistant turn
        prompt_with_assistant_turn = current_prompt_messages + [{"role": "assistant", "content": ""}]

        prompt_str = tokenizer.apply_chat_template(  # noqa: F821
            prompt_with_assistant_turn,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Format chosen and rejected responses (full conversation)
        chosen_str = tokenizer.apply_chat_template(  # noqa: F821
            current_chosen_messages,
            tokenize=False
        )
        rejected_str = tokenizer.apply_chat_template(  # noqa: F821
            current_rejected_messages,
            tokenize=False
        )

        # Tokenize (don't pad here, DataCollator will handle it)
        prompt_encoded = tokenizer(prompt_str, truncation=True, max_length=MAX_PROMPT_LENGTH)  # noqa: F821
        chosen_encoded = tokenizer(chosen_str, truncation=True, max_length=MAX_LENGTH)  # noqa: F821
        rejected_encoded = tokenizer(rejected_str, truncation=True, max_length=MAX_LENGTH)  # noqa: F821

        # Filter out examples that are too long after tokenization
        if (len(prompt_encoded['input_ids']) >= MAX_PROMPT_LENGTH or  # noqa: F821
            len(chosen_encoded['input_ids']) >= MAX_LENGTH or  # noqa: F821
            len(rejected_encoded['input_ids']) >= MAX_LENGTH):  # noqa: F821
            # print(f"Skipping example due to length: Prompt {len(prompt_encoded['input_ids'])}, Chosen {len(chosen_encoded['input_ids'])}, Rejected {len(rejected_encoded['input_ids'])}")
            continue

        processed["prompt_input_ids"].append(prompt_encoded["input_ids"])
        processed["chosen_input_ids"].append(chosen_encoded["input_ids"])
        processed["rejected_input_ids"].append(rejected_encoded["input_ids"])
        processed["prompt_attention_mask"].append(prompt_encoded["attention_mask"])
        processed["chosen_attention_mask"].append(chosen_encoded["attention_mask"])
        processed["rejected_attention_mask"].append(rejected_encoded["attention_mask"])
        processed["prompt_len"].append(len(prompt_encoded["input_ids"]))

    return processed



# Test Arena Hard Dataset Loading

We'll test different approaches to load the arena_hard dataset that has malformed JSON files.

In [ ]:

from datasets import load_dataset

dataset_id = "lmarena-ai/arena-hard-auto-v0.1"
dataset_split = "train"

dataset = load_dataset(dataset_id, split=dataset_split)


In [ ]:
# print a dataset sample
for i in range(100):
    assert len(dataset[i]['turns']) == 1 
    assert len(dataset[i]['turns'][0]) == 1
    print(dataset[i]['turns'][0]['content'])
    print("___"*30)

# Check eval results

In [ ]:
!pwd

In [ ]:
import json

# read json file
with open("out/evaluation_results_arena_hard.jsonl", "r") as f:
    results = [json.loads(line) for line in f]

In [ ]:
for res in results[:5]:
    print(res['response_pepo'])
    print("---"*30)
    print(res['response_dpo'])
    print("---"*30)
    print(res['judgment'])
    print("******"*30)